# 🏀 Hardwood: Grouping, Windows & Reshaping

### SIADS 505 · Other Assignments · Assignment 2
**100 points — 10 questions × 10 points each**

---

The box score is the oldest data product in sports. It is also, structurally,
a mess: percentages stored as `"7-13"`, playing time stored as `"26:38"`, and a
row for every player who dressed — including the three who never left the
bench.

This assignment is about the second half of pandas. Assignment 1 got you to
`groupby` and `pivot_table`. This one keeps going: **multi-level indexes**,
**named aggregation**, **window functions**, **ranking inside groups**, **runs
and streaks**, and the kind of string surgery that turns a scanned box score
into numbers you can actually add up.

> **Everything here is simulated.** Ten seasons, 320 games, 4,160 box-score
> lines, all generated from a fixed seed. The arenas are real; nothing else is.
> Do not cite any of it.

**How this is graded.** Ten questions, ten points each. The `assert`s printed
under each question are **worth zero points** — they exist so you can tell
whether you are on track. A separate set of hidden checks carries the grade,
and several of them re-run your function on a *different slice of the data*.
Write functions that compute the answer; do not paste in the numbers you see
below.

**Ground rules.** Every answer is a function that takes its data as arguments
and returns a value. Do not read files from inside a function unless the
question says to, do not modify the frame you were handed, and do not rename
the functions.


## The data

Three CSV files live in `data/`.

### `data/box_scores.csv` — one row per player per game, 4,160 rows

| column | what it is |
|---|---|
| `game_id` | `2019-G07` — season plus game number |
| `season` | 2015 through 2024 (the year the season *ends*) |
| `game_no` | 1–32 |
| `date` | game date |
| `player_id` | `P042`, stable across seasons |
| `player` | the player's name |
| `started` | `True` for the five who tipped off |
| `minutes` | `"26:38"` — **or the string `"DNP"`** |
| `field_goals` | `"7-13"`, made-attempted |
| `three_pointers` | `"2-5"` |
| `free_throws` | `"4-4"` |
| `offensive_rebounds`, `defensive_rebounds` | |
| `assists`, `turnovers`, `steals`, `blocks`, `fouls` | |
| `points` | equals `2×FGM + 3PM + FTM` |

**1,134 of those rows are `DNP`** — the player dressed and never checked in.
Everything but `game_id`, `season`, `game_no`, `date`, `player_id`, `player`,
`started` and `minutes` is blank on those rows. A DNP is not a zero. A player
who did not play did not shoot 0-for-0; he did not shoot.

### `data/games.csv` — one row per game, 320 rows

`game_id`, `season`, `game_no`, `date`, `opponent`, `conference`, `site`
(`Home`/`Away`/`Neutral`), `venue`, `team_points`, `opponent_points`,
`overtime` (0 or 1), `attendance` (**9 missing**).

`team_points` is exactly the sum of the box score for that game. If your
numbers disagree, your numbers are wrong.

### `data/roster.csv` — one row per player per season, 150 rows

`season`, `player_id`, `player`, `jersey`, `position` (`G`/`F`/`C`), `class`
(`Fr`/`So`/`Jr`/`Sr`), `height_in`, `hometown`, `dressed`.

Thirteen players a year suit up; two more are on the roster and never appear in
a box score. Join on **both** `season` and `player_id` — a `player_id` shows up
in as many as four different seasons, in a different class each time.


---

## Setup

Run this first.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print("Tip-off.")


### First contact

Look at the thing before you write anything. This cell is ungraded — poke at
it, add cells, break it. Pay attention to what `dtypes` says about the columns
that *look* numeric.


In [ ]:
raw = pd.read_csv("data/box_scores.csv")

print(raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("dtype"))
print("DNP rows:", (raw["minutes"] == "DNP").sum())
display(raw.loc[raw["minutes"] == "DNP"].head(3))


---

## Question 1 — Tip-off  ·  10 points

Two loaders, and one of them wants a **MultiIndex**.

Write **`load_box_scores(path="data/box_scores.csv")`** so that it:

1. reads the CSV at `path`,
2. parses **`date`** as a real datetime,
3. sets **`["game_id", "player_id"]`** as the index — in that order,
4. and returns it **sorted by the index**.

The result is a `DataFrame` with **4,160 rows and 17 columns** — 17, because
two of the nineteen columns moved into the index.

Then write **`load_games(path="data/games.csv")`**, which parses `date` and
sets `game_id` as a plain single-level index: **320 rows, 11 columns**.

*Hint:* `set_index` accepts a list. `.sort_index()` matters — a lot of what
comes later is faster and less surprising on a sorted index, and the hidden
checks look for it.


In [ ]:
def load_box_scores(path="data/box_scores.csv"):
    """Load box_scores.csv, dates parsed, indexed by (game_id, player_id)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def load_games(path="data/games.csv"):
    """Load games.csv with a datetime `date` column, indexed by `game_id`."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
box = load_box_scores()
games = load_games()

assert isinstance(box, pd.DataFrame), "load_box_scores should return a DataFrame"
assert box.shape == (4160, 17), f"expected (4160, 17), got {box.shape}"
assert isinstance(box.index, pd.MultiIndex), "the index should be a MultiIndex"
assert list(box.index.names) == ["game_id", "player_id"], box.index.names
assert box.index.is_monotonic_increasing, "sort the index"
assert pd.api.types.is_datetime64_any_dtype(box["date"]), "`date` should be datetime"

assert games.shape == (320, 11), f"expected (320, 11), got {games.shape}"
assert games.index.name == "game_id"
assert games["team_points"].sum() == 26301, "these are the simulated totals"

# One game is one slice of the MultiIndex.
opener = box.loc["2015-G01"]
assert len(opener) == 13, "thirteen players dressed for every game"

print("✅ Q1 public checks passed — you are on the floor.")
box.head()


---

## Question 2 — Splitting the line  ·  10 points

`field_goals` reads `"7-13"`. That is two numbers wearing a trench coat.

Write **`split_shooting(box)`** returning a **new** DataFrame with the **same
index** as `box` and exactly these six columns, in this order:

| column | from |
|---|---|
| `fg_made`, `fg_att` | `field_goals` |
| `fg3_made`, `fg3_att` | `three_pointers` |
| `ft_made`, `ft_att` | `free_throws` |

All six are **floats**, because the 1,134 DNP rows have no number at all and
must come back as **`NaN`** — not `0`.

*Hint:* `.str.split("-", expand=True)` turns one string column into two, and
`pd.to_numeric(..., errors="coerce")` turns the pieces into numbers while
leaving the blanks as `NaN`.


In [ ]:
def split_shooting(box):
    """Split the made-attempted strings into six float columns."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
shots = split_shooting(box)

assert list(shots.columns) == ["fg_made", "fg_att", "fg3_made", "fg3_att", "ft_made", "ft_att"]
assert len(shots) == 4160 and shots.index.equals(box.index), "keep every row, keep the index"
assert all(pd.api.types.is_float_dtype(shots[c]) for c in shots.columns), "floats, not ints"

dnp = box["minutes"] == "DNP"
assert shots[dnp].isna().all().all(), "a DNP is NaN across all six columns"
assert shots[~dnp].notna().all().all(), "rows that played should never be NaN"

assert shots["fg_att"].sum() == 18039.0
assert shots["fg_made"].sum() == 9599.0
played_rows = shots.dropna()
assert (played_rows["fg3_att"] <= played_rows["fg_att"]).all(), "a three is a field goal attempt too"

made, att = shots["fg_made"].sum(), shots["fg_att"].sum()
print(f"🎯 Ten simulated seasons: {made:,.0f}-for-{att:,.0f} from the field ({made / att:.1%}).")
print("✅ Q2 public checks passed.")
shots.head()


---

## Question 3 — Stopping the clock  ·  10 points

`minutes` reads `"26:38"` — twenty-six minutes and thirty-eight seconds — or
the string `"DNP"`.

Write **`minutes_played(box)`** returning a float **Series**:

- named **`minutes_played`**,
- same index, all **4,160** rows,
- `"26:38"` → **`26.63`** (minutes plus seconds ÷ 60, **rounded to 2 decimals**),
- `"DNP"` → **`NaN`**.

Throwing away the seconds is the trap here. Over a season it is worth several
whole games of playing time, and the hidden checks look specifically at the
rows where the clock did not land on a round minute.

*Hint:* `.str.split(":", expand=True)` gives you two pieces; coerce both with
`pd.to_numeric(..., errors="coerce")` and the `"DNP"` rows take care of
themselves.


In [ ]:
def minutes_played(box):
    """Turn the MM:SS `minutes` column into float minutes."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
mins = minutes_played(box)

assert isinstance(mins, pd.Series) and mins.name == "minutes_played"
assert pd.api.types.is_float_dtype(mins), f"expected floats, got {mins.dtype}"
assert len(mins) == 4160 and mins.isna().sum() == 1134, "the DNP rows are NaN"
assert abs(mins.sum() - 64550.03) < 0.05
assert abs(mins.max() - 41.12) < 0.01, "overtime is a real thing"

fractional = (mins.dropna() % 1 != 0).mean()
assert fractional > 0.9, "you dropped the seconds — 26:38 is not 26 minutes"

print(f"⏱️  {mins.sum():,.0f} player-minutes across ten seasons.")
print(f"   Longest night: {mins.idxmax()} — {mins.max():.2f} minutes.")
print("✅ Q3 public checks passed.")
mins.head()


---

## Question 4 — Shooting rates  ·  10 points

Now divide — carefully. A player who never took a three did not shoot 0%; he
has no three-point percentage at all.

Write **`shooting_rates(box)`** returning a DataFrame on the **same index**
with these four columns, in this order, each **rounded to 3 decimals**:

| column | formula |
|---|---|
| `fg_pct` | `fg_made / fg_att` |
| `fg3_pct` | `fg3_made / fg3_att` |
| `ft_pct` | `ft_made / ft_att` |
| `true_shooting` | `points / (2 × (fg_att + 0.44 × ft_att))` |

**Wherever the denominator is zero, the answer is `NaN`** — never `0.0`, never
`inf`. DNP rows are already `NaN` and stay that way.

True shooting can legitimately come out **above 1.0** (three-pointers and free
throws both beat two points per shot). That is not a bug — do not clip it.

*Hint:* you may call your own `split_shooting` in here. `.replace(0, np.nan)`
on the denominator is the cleanest way to make a zero denominator produce
`NaN` instead of `inf`.


In [ ]:
def shooting_rates(box):
    """FG%, 3P%, FT% and true shooting, with NaN wherever nothing was attempted."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
rates = shooting_rates(box)

assert list(rates.columns) == ["fg_pct", "fg3_pct", "ft_pct", "true_shooting"]
assert len(rates) == 4160 and rates.index.equals(box.index)
assert not np.isinf(rates.to_numpy(dtype=float)).any(), "dividing by zero produced inf"

shots = split_shooting(box)
assert rates.loc[shots["fg3_att"] == 0, "fg3_pct"].isna().all(), "no attempts, no percentage"
assert rates["fg_pct"].isna().sum() == 1248
assert rates["true_shooting"].isna().sum() == 1208

assert abs(rates["fg_pct"].mean() - 0.5016) < 0.001
assert abs(rates["true_shooting"].mean() - 0.6133) < 0.001
assert rates["true_shooting"].max() > 1.0, "true shooting above 1.0 is allowed"

print(f"🏹 Mean true shooting on nights a player got a shot up: {rates['true_shooting'].mean():.3f}")
print("✅ Q4 public checks passed.")
rates.head()


---

## Question 5 — The season totals  ·  10 points

`groupby` with **named aggregation** — `.agg(new_name=("column", "how"))` — is
how you build a summary table without a second pass to rename everything.

Write **`player_season_totals(box)`**. **Drop the DNP rows first**, then group
by `season` and `player` and return a DataFrame with a `(season, player)`
**MultiIndex**, sorted, and these columns in this order:

| column | definition |
|---|---|
| `games_played` | games in which the player actually played |
| `minutes` | total minutes, **rounded to 1 decimal** |
| `points` | total points, as an **int** |
| `rebounds` | offensive **plus** defensive, as an **int** |
| `assists` | total assists, as an **int** |
| `ppg` | `points / games_played`, **rounded to 1** |

130 rows come back — one per player per season in which they appeared.

*Hint:* `.agg(games_played=("points", "size"), points=("points", "sum"), ...)`.
Add the minutes and combined-rebounds columns to a copy of the filtered frame
before you group.


In [ ]:
def player_season_totals(box):
    """One row per player per season: games, minutes, points, rebounds, assists, ppg."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
totals = player_season_totals(box)

assert list(totals.columns) == ["games_played", "minutes", "points", "rebounds", "assists", "ppg"]
assert list(totals.index.names) == ["season", "player"]
assert totals.shape == (130, 6), f"expected (130, 6), got {totals.shape}"
assert totals["games_played"].max() == 32, "nobody plays more than 32 games"
assert totals["points"].dtype.kind == "i", "points should be a whole number"

# The totals must reconcile with the game log.
assert totals.groupby("season")["points"].sum().sum() == games["team_points"].sum()

best = totals["ppg"].idxmax()
print(f"🔥 Best simulated scoring season: {best[1]} in {best[0]} — {totals['ppg'].max()} ppg")
print("✅ Q5 public checks passed.")
totals.head()


---

## Question 6 — Who led the team tonight  ·  10 points

One row per game, naming the player who scored the most. The interesting part
is the tie: on plenty of nights two players finish level.

Write **`leading_scorer(box)`** returning a DataFrame indexed by **`game_id`**
(320 rows, sorted) with columns **`["player", "points"]`**, `points` as an
**int**. Ignore DNP rows.

**Tie-break, in order:**

1. most points,
2. then whoever played **more minutes**,
3. then, if it is still tied, the player whose name comes **first
   alphabetically**.

*Hint:* `sort_values` takes a list of columns and a matching list of
`ascending=` flags. Once the frame is in the right order,
`groupby("game_id").head(1)` takes the winner of each game. You will need
minutes as a number — call your `minutes_played`.


In [ ]:
def leading_scorer(box):
    """The top scorer in every game, one row per game_id."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
leaders = leading_scorer(box)

assert list(leaders.columns) == ["player", "points"]
assert leaders.index.name == "game_id"
assert len(leaders) == 320 and leaders.index.is_unique, "exactly one row per game"
assert leaders["points"].dtype.kind == "i"

# Nobody can lead a game with fewer points than someone else scored in it.
played = box[box["minutes"] != "DNP"]
assert (leaders["points"] == played.groupby("game_id")["points"].max()).all()

assert leaders.loc["2015-G01", "player"] == "Moussa Novotny"
assert leaders.loc["2015-G01", "points"] == 25

top = leaders["player"].value_counts()
print("🥇 Most games led in scoring:")
print(top.head(3).to_string())
print("✅ Q6 public checks passed.")
leaders.head()


---

## Question 7 — Rolling form  ·  10 points

"They're playing better lately" is a claim about a **window**, not a season.

Write **`rolling_form(games, window=5)`** returning a float **Series** named
**`rolling_points`**:

- the mean of `team_points` over the current game and the `window - 1` before
  it, **rounded to 2 decimals**,
- computed **within each season** — game 1 of a new year must not average in
  last March,
- games ordered by `season` then `game_no`,
- **indexed exactly like `games`, in `games`' original row order**,
- `NaN` for the first `window - 1` games of each season (40 of them at
  `window=5`).

The `window` argument has to work: the hidden checks call it with `10`.

*Hint:* `games.sort_values([...]).groupby("season")["team_points"].rolling(window).mean()`
gives you a Series with a *two-level* index — season on the outside. Peel that
level off with `.droplevel("season")`, then `.reindex(games.index)` to put the
rows back the way you found them.


In [ ]:
def rolling_form(games, window=5):
    """Rolling mean of team_points over the last `window` games of that season."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
form = rolling_form(games)

assert isinstance(form, pd.Series) and form.name == "rolling_points"
assert len(form) == 320 and form.index.equals(games.index), "same index, same order"
assert form.isna().sum() == 40, "four warm-up games in each of ten seasons"
assert form[games["game_no"] <= 4].isna().all(), "the window restarts every season"

assert abs(form.loc["2015-G05"] - 83.0) < 0.01
assert abs(form.max() - 97.6) < 0.01 and form.idxmax() == "2019-G25"

wider = rolling_form(games, window=10)
assert wider.isna().sum() == 90, "window= has to actually do something"

print(f"📈 Hottest simulated five-game stretch ended {form.idxmax()}: {form.max():.1f} ppg.")
print("✅ Q7 public checks passed.")
form.dropna().head()


---

## Question 8 — The leaderboard  ·  10 points

Ranking *inside* a group is the thing people reach for `sort_values` for and
then get wrong. `.rank()` does it in one call.

Write **`season_leaderboard(box, min_games=20)`**:

1. start from the same season totals as Question 5,
2. keep only players with **`games_played >= min_games`**,
3. add a **`rank`** column: the player's position by `ppg` **within their own
   season**, best = 1, using **`method="min"`** so that a tie gives both
   players the better rank and skips the next one — stored as an **int**,
4. return columns **`["games_played", "points", "ppg", "rank"]`**, on the same
   `(season, player)` MultiIndex,
5. sorted by `season` ascending, then `rank` ascending, then `player`
   alphabetically.

At `min_games=20` that is 81 rows. The hidden checks also call it with
`min_games=28`.

*Hint:* `qualified.groupby("season")["ppg"].rank(method="min", ascending=False)`.
`groupby(...).rank()` returns one value per row, aligned to the frame you
started with — no merge needed.


In [ ]:
def season_leaderboard(box, min_games=20):
    """Qualified scorers ranked within each season."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
board = season_leaderboard(box)

assert list(board.columns) == ["games_played", "points", "ppg", "rank"]
assert list(board.index.names) == ["season", "player"]
assert board.shape == (81, 4), f"expected (81, 4) at min_games=20, got {board.shape}"
assert board["rank"].dtype.kind == "i", "rank should be a whole number"
assert (board.reset_index().groupby("season")["rank"].min() == 1).all(), "rank restarts each season"

assert board.loc[(2015, "Moussa Novotny"), "rank"] == 1
assert board.loc[(2024, "Zavier Lindqvist"), "rank"] == 1
assert season_leaderboard(box, min_games=28).shape[0] == 56, "min_games= has to do something"

print("🏅 Simulated scoring champion, by season:")
print(board[board["rank"] == 1].reset_index()[["season", "player", "ppg"]].to_string(index=False))
print("✅ Q8 public checks passed.")
board.head()


---

## Question 9 — The longest streak  ·  10 points

Streaks are the classic "pandas can't do that" problem. Pandas can do that.

Write **`longest_win_streak(games)`** returning an **int Series**:

- named **`longest_win_streak`**,
- indexed by **`season`** (index named `season`), ascending,
- one row for **every season present in the input**,
- the value is the longest run of **consecutive wins**, with games ordered by
  `game_no` inside the season.

A season with **no wins at all** is a streak of **`0`** — a row with a zero in
it, not a missing row. The hidden checks hand you a tiny winless fixture on
purpose.

*Hint:* the standard trick. Build a boolean `won`. A new run starts wherever
the value differs from the row before it, so `(won != won.shift()).cumsum()`
labels each run with a number. Count the rows in each *winning* run, take the
biggest per season, and `reindex` so the seasons that never won come back as
`0`.


In [ ]:
def longest_win_streak(games):
    """Longest run of consecutive wins in each season."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
streaks = longest_win_streak(games)

assert isinstance(streaks, pd.Series) and streaks.name == "longest_win_streak"
assert streaks.index.name == "season"
assert list(streaks.index) == list(range(2015, 2025))
assert streaks.dtype.kind == "i", "a streak is a whole number of games"

assert streaks.loc[2019] == 17, "the 2019 team went on a run"
assert streaks.loc[2023] == 3

# A season with nothing to celebrate is a zero, not a missing row.
winless = pd.DataFrame(
    {"season": [1999, 1999], "game_no": [1, 2],
     "team_points": [60, 61], "opponent_points": [70, 71]},
    index=pd.Index(["x", "y"], name="game_id"),
)
assert longest_win_streak(winless).loc[1999] == 0

print("🔥 Longest simulated winning streak by season:")
print(streaks.to_string())
print("✅ Q9 public checks passed.")


---

## Question 10 — Class and position  ·  10 points

Last one. Merge on two keys, pivot, and then put the rows in an order pandas
would never pick for you.

Write **`class_scoring(box, roster)`**:

1. drop the DNP rows,
2. **merge** with `roster` on **both** `season` and `player_id`, `how="left"`
   — a `player_id` is in the roster once per season, in a different class each
   time, so a single-key merge on `player_id` alone will multiply your rows,
3. **pivot** to the **mean `points` per game played**,
4. rows: `class`, in **academic order — `Fr`, `So`, `Jr`, `Sr`** — not
   alphabetical,
5. columns: `position`, in the order **`G`, `F`, `C`**,
6. values **rounded to 2 decimals**, index named `class`, columns named
   `position`.

*Hint:* `pivot_table` sorts both axes alphabetically, which gets you
`Fr, Jr, So, Sr` and `C, F, G`. Fix it afterwards with
`.reindex(index=[...], columns=[...])`, which also guarantees the shape even if
a combination never happened.


In [ ]:
def class_scoring(box, roster):
    """Mean points per game by academic class (rows) and position (columns)."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
roster = pd.read_csv("data/roster.csv")
table = class_scoring(box, roster)

assert list(table.index) == ["Fr", "So", "Jr", "Sr"], "academic order, not alphabetical"
assert list(table.columns) == ["G", "F", "C"]
assert table.index.name == "class" and table.columns.name == "position"

assert abs(table.loc["Sr", "F"] - 12.57) < 0.01
assert abs(table.loc["Fr", "G"] - 4.30) < 0.01
assert (table.loc["Sr", ["G", "F"]] > table.loc["Fr", ["G", "F"]]).all(),     "seniors outscore freshmen in the backcourt and at forward"
assert table.loc["Fr", "C"] > table.loc["Sr", "C"],     "...but not at center, which is the interesting part of this table"

print("📊 Mean points per game played, by class and position:")
print(table.to_string())
print("✅ Q10 public checks passed — that's all ten. Go Blue. 〽️")
table


---

# 🎉 Victory lap

That's 100 points. Everything below is **ungraded** — no asserts, no hidden
tests, nothing to hand in. It is here because this is the part of the job that
is actually fun.


In [ ]:
# Ten seasons at a glance, with the rolling five-game form at the end of each.
g = games.copy()
g["won"] = g["team_points"] > g["opponent_points"]
g["margin"] = g["team_points"] - g["opponent_points"]

g.groupby("season").agg(
    wins=("won", "sum"),
    losses=("won", lambda s: (~s).sum()),
    ppg=("team_points", "mean"),
    opp_ppg=("opponent_points", "mean"),
    avg_margin=("margin", "mean"),
    overtimes=("overtime", "sum"),
).round(1)


In [ ]:
# Does the roll carry? Rolling form going in, versus the margin that night.
form = rolling_form(games)
carry = games.assign(form=form, margin=games["team_points"] - games["opponent_points"]).dropna(subset=["form"])
print("correlation between five-game form and that night's margin:",
      round(carry["form"].corr(carry["margin"]), 3))

carry["form_bucket"] = pd.qcut(carry["form"], 4, labels=["cold", "cool", "warm", "hot"])
carry.groupby("form_bucket", observed=True)["margin"].agg(["size", "mean"]).round(2)


In [ ]:
# Career arcs: the same player, season by season.
totals = player_season_totals(box)
careers = totals.reset_index().groupby("player").filter(lambda d: len(d) >= 4)
careers.pivot_table(index="player", columns="season", values="ppg").dropna(thresh=4).round(1)


In [ ]:
# Efficiency versus volume — the argument every fan base has.
rates = shooting_rates(box)
mins = minutes_played(box)
usage = box.assign(ts=rates["true_shooting"], mp=mins).dropna(subset=["ts"])

usage.groupby("player").agg(
    games=("ts", "size"),
    mpg=("mp", "mean"),
    ppg=("points", "mean"),
    ts=("ts", "mean"),
).query("games >= 60").round(3).sort_values("ppg", ascending=False).head(12)


In [ ]:
# A picture, if matplotlib is installed.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed — try: pip install matplotlib")
else:
    form = rolling_form(games)
    fig, ax = plt.subplots(figsize=(11, 4))
    for season, chunk in games.assign(form=form).groupby("season"):
        chunk = chunk.sort_values("game_no")
        ax.plot(chunk["game_no"], chunk["form"], color="#00274C", alpha=0.45, linewidth=1.2)
    best = games.assign(form=form).query("season == 2019").sort_values("game_no")
    ax.plot(best["game_no"], best["form"], color="#FFCB05", linewidth=2.8, label="2019")
    ax.set_title("Simulated five-game rolling scoring, all ten seasons")
    ax.set_xlabel("game number")
    ax.set_ylabel("points per game (5-game window)")
    ax.legend()
    plt.tight_layout()
    plt.show()


---

### Your turn

Add a cell and answer something nobody asked you to. Things this data can
actually support:

- Do **starters** shoot better than the bench, once you control for minutes?
- Is there a **freshman wall** — do first-year players fall off after January?
  (`.dt.month` on `date`)
- Which player-season had the best **assist-to-turnover ratio**, and how many
  games does a player need before that number means anything?
- Do **overtime** games cluster against particular conferences?
- Nine games are missing `attendance`. What is the least dishonest way to fill
  them, and what does your choice do to the answer you get?
- Reshape `box_scores` from one-row-per-player-per-game into a **wide** frame
  of players × games with `pivot_table`. What did you lose?

---

<div align="center">

**〽️ Go Blue.**

*Hail to the victors, and to everyone who checked `.dtypes` before trusting a column.*

</div>
